In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.optim as optim
import torch.nn as nn
import skfuzzy as fuzz
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from ANFISpy import ANFIS, LSTMANFIS
from ANFISpy import HamacherAND, ProdAND

In [2]:
project_root = Path.cwd().parent  # Go up one level from /notebooks/ to the project root
data_path = project_root / "Data" / "Processed Data" / "AirQualityUCI_cleaned.csv"

# --- Load the Dataset ---
df = pd.read_csv(data_path)

# --- Features & Target ---
X = df[['PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)',
        'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']].values

y = df['C6H6(GT)'].values

time = df.index.values  # or df['DateTime'] if you have a timestamp column

# --- Preview the Data ---
df.head()

,Date,Time,PT08.S1(CO),C6H6(GT),PT08.S2(NMHC),PT08.S3(NOx),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,2004-03-10,18:00:00,1360.00,11.881723,1045.50,1056.25,1692.00,1267.50,13.60,48.875001,0.757754
1,2004-03-10,19:00:00,1292.25,9.397165,954.75,1173.75,1558.75,972.25,13.30,47.700000,0.725487
2,2004-03-10,20:00:00,1402.00,8.997817,939.25,1140.00,1554.50,1074.00,11.90,53.975000,0.750239
3,2004-03-10,21:00:00,1375.50,9.228796,948.25,1092.00,1583.75,1203.25,11.00,60.000000,0.786713
4,2004-03-10,22:00:00,1272.25,6.518224,835.50,1205.00,1490.00,1110.00,11.15,59.575001,0.788794


In [3]:
# --- Sequence Construction ---
x, y_seq, t = [], [], []
seq_len = 10  # number of timesteps per input sequence

for i in range(len(X) - seq_len):
    x.append(X[i : i + seq_len])       # sequence of past features
    y_seq.append(y[i + seq_len])       # next-step target
    t.append(time[i + seq_len])        # corresponding timestamp

x = torch.FloatTensor(np.array(x))       # shape: (samples, seq_len, n_features)
y_seq = torch.FloatTensor(np.array(y_seq)).unsqueeze(-1)  # shape: (samples, 1)
t = np.array(t)

# --- Train / Val / Test Split ---
train_size = int(0.7 * len(x))
val_size = int(0.15 * len(x))

x_train = x[:train_size]
y_train = y_seq[:train_size]
t_train = t[:train_size]

x_val = x[train_size:train_size + val_size]
y_val = y_seq[train_size:train_size + val_size]
t_val = t[train_size:train_size + val_size]

x_test = x[train_size + val_size:]
y_test = y_seq[train_size + val_size:]
t_test = t[train_size + val_size:]

# --- Sanity check ---
print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_val:   {x_val.shape},   y_val:   {y_val.shape}")
print(f"x_test:  {x_test.shape},  y_test:  {y_test.shape}")

x_train: torch.Size([6542, 10, 8]), y_train: torch.Size([6542, 1])
x_val:   torch.Size([1402, 10, 8]),   y_val:   torch.Size([1402, 1])
x_test:  torch.Size([1403, 10, 8]),  y_test:  torch.Size([1403, 1])


In [4]:
n_vars = 8
mf_names = ['L', 'M', 'H']  # 3 MFs per variable

variables = {
    'inputs': {
        'n_sets': [3] * n_vars,                # 3 MFs per feature
        'uod': [(0, 1)] * n_vars,              # assuming standardized inputs
        'var_names': ['PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)',
                      'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH'],
        'mf_names': [list(mf_names) for _ in range(n_vars)],  # unique MF labels per input
    },
    'output': {
        'var_names': 'C6H6(GT)',
        'n_classes': 1,
    },
}

lstmanfis = LSTMANFIS(variables, 'bell', seq_len, output_activation=nn.Identity())

In [5]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lstmanfis.parameters(), lr=0.01)

epochs = 10
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs, _ = lstmanfis(x_train)
    loss_train = criterion(outputs[:, -1, :] , y_train)
    loss_train.backward()
    optimizer.step()
    
    lstmanfis.eval()
    with torch.no_grad():
        outputs, _ = lstmanfis(x_val)
        loss_val = criterion(outputs[:, -1, :] , y_val)
    
    if epoch % 2 == 0:
        print(f'Epoch {epoch} | Train Loss: {loss_train.item():.5f} | Validation Loss: {loss_val.item():.5f}')

RuntimeError: [enforce fail at alloc_cpu.cpp:114] data. DefaultCPUAllocator: not enough memory: you tried to allocate 35074654208 bytes.

In [ ]:
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss Curves')
plt.show()